# Choosing how many epochs to train the base model

### Imports

In [1]:
import sys
print(sys.version)

3.9.25 (main, Apr 17 2026, 00:00:00) 
[GCC 11.5.0 20240719 (Red Hat 11.5.0-14)]


In [2]:
import os
import json

In [3]:
%ls

data/                              __pycache__/
evaluation/                        README.md
master_auditor.ipynb               results/
master_hyperparams.py              results_to_replicate.txt
master_pretraining.ipynb           trainer/
master_retrain_from_scratch.ipynb  unlearn/
master_unlearning.ipynb            visualize_pretraining_results.ipynb
models/                            visualize_results.ipynb
_old/                              wandb/


In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
# import matplotlib.pyplot as plt

# from trainer.utils import training_regimen_lr_annealing


### Set configs for the pretraining

In [5]:

from master_hyperparams import hyperparams

device = "cuda" if torch.cuda.is_available() else "cpu"

dataset = "CIFAR10"
model_class = "AllCNN"
hp = hyperparams[dataset]
model_hp = hp[model_class]

pretraining_config = {

    "description": "Pre-training - AllCNN, CIFAR10",
    
    "device": device,
    "model_class": model_class,
    "data": {
        "dataset": dataset,
        "num_classes": hp["num_classes"],
        "batch_size": 1024,  # larger batch for faster pretraining
        "num_workers": hp["num_workers"],
        },

    "training": {
        "num_epochs": [1, 20, 30, 50, 75, 100],
        "num_runs": 3,
        "learning_rate": model_hp["training"]["learning_rate"],
        "weight_decay": model_hp["training"]["weight_decay"],
        "batch_print_freq": 12,
        },
}


### Protocol for several runs

In [6]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /cs/student/msc/ml/2025/jmoncus/.netrc.
wandb: Currently logged in as: jjmoncus (jjmoncus706) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
import os
import json
import random
import glob
import torch
import wandb
import torch.nn as nn
import torch.optim as optim
from models.archs.utils import init_model
from data.dataloaders import load_dataloaders_for_experiment
from trainer.utils import init_folder_if_not_exists, training_regimen_lr_annealing
from trainer.val import validate
from models.archs.utils import init_model
from data.utils import setup_seed

def run_pretraining(config):

    print("-"*75)
    print("-"*13 + "  " + f"EVALUATING # OF EPOCHS FOR TRAINING {config['model_class']}" + "  " + "-"*13)
    print("-"*75 + "\n")

    setup_seed(config["GRAND_SEED"])

    # init wandb
    wandb.init(
      project="Verifying-Unlearning-2026",
      name=f"Pretraining Experiments - {config['model_class']}",
      config=config,
      reinit="finish_previous"
    )

    # Make experiment results folder if it doesnt already exist
    results_folder = init_folder_if_not_exists( f"results/seed_{config['GRAND_SEED']}/pretraining" )
    
    # Save the config for this experiment to the main results folder
    with open(os.path.join(results_folder, "experiment_config.json"), "w") as f:
        json.dump(config, f, indent=4)

    # make model checkpoints folder for this seed if it doesn't exist yet
    checkpoints_folder = init_folder_if_not_exists( f"models/model_checkpoints/seed_{config['GRAND_SEED']}/pretrained" )


    for num_epochs in config["training"]["num_epochs"]:

        epoch_results_folder = init_folder_if_not_exists( os.path.join(results_folder, f"{config['data']['dataset']}_{config['model_class']}_{num_epochs}_epochs") )
        epoch_checkpoints_folder = init_folder_if_not_exists( os.path.join(checkpoints_folder, f"{config['data']['dataset']}_{config['model_class']}_{num_epochs}_epochs") )


        # get some data (does not change in between runs)
        train_loader, _, test_loader = load_dataloaders_for_experiment(
            name = config["data"]["dataset"],
            batch_size=config["data"]["batch_size"], 
            num_workers=config["data"]["num_workers"], 
            seed=config["GRAND_SEED"], 
            class_to_replace=None, 
            percent_to_replace=None,
            val=False
            )

        # ----------------------------------------------------------------------------------- #
        # ----------------------------------------------------------------------------------- #
        # ---------------- TRAIN A BASE MODEL, FROM WHICH UNLEARNING BEGINS ----------------- #
        # ----------------------------------------------------------------------------------- #
        # ----------------------------------------------------------------------------------- #


        print("-"*55)
        print("-"*13 + "  " + f"TESTING: {num_epochs} EPOCHS" + "  " + "-"*13)
        print("-"*55 + "\n")

        for i in range(1, config["training"]["num_runs"] + 1):

            # init model, opt, criterion, and scheduler
            empty_model = init_model(model_class = config["model_class"], num_classes=config["data"]["num_classes"]).to(config["device"])
            criterion = nn.CrossEntropyLoss()
            opt = optim.Adam(empty_model.parameters(), lr=config["training"]["learning_rate"], weight_decay = config["training"]["weight_decay"])
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                opt, 
                T_max=num_epochs, 
                eta_min=1e-6
            )

            # train (wandb logging underneath, dont need to re-log training accuracy)
            base_model_path = os.path.join(epoch_checkpoints_folder, f"{config['model_class']}_{i}.pth")
            trained_model, opt, scheduler, _, _, _, _ = training_regimen_lr_annealing(
                empty_model, 
                train_loader,
                opt, 
                criterion, 
                scheduler, 
                device = config["device"], 
                num_epochs=num_epochs, 
                model_path = base_model_path,
                print_freq = config["training"]["batch_print_freq"])

            # evaluate trained model on some test
            print(f"Evaluating model trained for {num_epochs} epochs on test set...\n") 
            train_out = validate(
                train_loader, 
                trained_model, 
                criterion, 
                print_freq = config["training"]["batch_print_freq"],
                device = config["device"]
            )
            
            test_out = validate(
                test_loader, 
                trained_model, 
                criterion, 
                print_freq = config["training"]["batch_print_freq"],
                device = config["device"]
            )

            print(f"Train accuracy: {train_out['avg_acc']:.4f}\n")
            print(f"Test accuracy: {test_out['avg_acc']:.4f}\n")
            
            results = {
                "num_epochs": num_epochs,
                "train_acc": train_out["avg_acc"],
                "test_acc": test_out["avg_acc"]
                }

            # save results
            with open(os.path.join(epoch_results_folder, f"results_{i}.json"), "w") as f:
                json.dump(results, f, indent=4)

            # the checkpoint saving is handled by 'training_w_lr_annealing' above
                    
    wandb.finish()

    print("-"*70)
    print("-"*19 + "  " + f"FINISHED PRETRAINING" + "  " + "-"*19)
    print("-"*70 + "\n")

### Check metrics on unlearned models

In [8]:
# MAKE A RANDOM SEED
pretraining_config["GRAND_SEED"] = 11
# DO EXP
run_pretraining(config = pretraining_config)

---------------------------------------------------------------------------
-------------  EVALUATING # OF EPOCHS FOR TRAINING AllCNN  -------------
---------------------------------------------------------------------------

setup random seed = 11


results/seed_11/pretraining doesn't exist - creating it...

models/model_checkpoints/seed_11/pretrained doesn't exist - creating it...

results/seed_11/pretraining/CIFAR10_AllCNN_1_epochs doesn't exist - creating it...

models/model_checkpoints/seed_11/pretrained/CIFAR10_AllCNN_1_epochs doesn't exist - creating it...

========== DATALOADER INFO
Dataset: CIFAR-10
Train: 50000 images for training
Test: 10000 images for testing
Training augmentation = randomcrop(32,4) + randomhorizontalflip + colorjitter + randomrotation + normalize
Validation/Test augmentation = normalize
num_workers = 4


-------------------------------------------------------
-------------  TESTING: 1 EPOCHS  -------------
-------------------------------------------------------

 ----- EPOCH 1 ----- 

Epoch: [1][11/49]	Loss 1.8157 (1.9881)	Accuracy 33.203 (25.301)	Entropy 1.9706 (2.0886)	M-Entropy 1.6655 (1.8464)	Time 4.20
Epoch: [1][23/49]	Loss 1.6229 (1.8485)	Accuracy 40.820 (31.144)	Entropy 1.7739 (1.9746)	M-Entropy

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


val_accuracy 91.680

Train accuracy: 99.7180

Test accuracy: 91.6800



RAM_GB,▇▇▇▇▇▁█▁▁▁▂▁▁▁▁▁▂▁▂▂████▁▁▁▁▅▅▅▁▂▁▁▁▁▂▂▁
VRAM_GB,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▂▂▁▃▁▁▁▁▃▃▁▃▃▄▅▁▂▄▆▆▁▃▃▁▇▃▅▇▇▇▃▃▃█▃▃▄█
learning_rate,▂█▄▂▆▆▆▁▁▂▄▄▁██▇▄▃▇▆▆▄▃▂▂▁▇▃▃▁██▆▅▃▁▁██▂
time (batch),█▄▁▁▁▁▁▁▁▄▁▄▁▄▁▁▂▁▂▁▂▄▁▄▂▁▁▁▁▄▂▄▄▂▁▁▂▁▁▁
train_acc (batch),▇▂▆▇▇▁▄▇█▅▇▇▇▇█▆▇███▇▇▇▇▆██████▅▆▇▇█▇▇▇█
train_acc (full),▅▁▆▆▇▃▃▄▇▄▇██▄▇▇▇▇██████▁███▆▆▇▇███▇████
train_entropy (batch),▆▄▂▇▃▂▄▄▆▂▅▂▂▁▂▁█▄▂▁▃▂▂▂▁▂▂▁▁▁▁▃▁▁▇▃▁▁▁▁
train_entropy (full),█▃▂▆▅▂▄▂▆▂▄▃▃█▆▃▂▅▂▄▂▁▁▅▄▁▁▁▄▂▁▁▁▁▁▁▃▂▂▁
train_loss (batch),█▃▃▂▂▂▂▃▃▂▅▂▂▆▃▃▁▄▂▂▁▃▂▁▁▂▂▂▁▁▁▁▁▁▃▂▁▁▁▁
+4,...


----------------------------------------------------------------------
-------------------  FINISHED PRETRAINING  -------------------
----------------------------------------------------------------------

